In [43]:
# ==========================================
# BLOQUE DE CÓDIGO: Preparación del Corpus 
# ==========================================

import pandas as pd
import numpy as np

In [33]:
# Llamamos a la función (usará la ruta por defecto de Windows que ya configuraste)
def cargar_y_preprocesar_corpus(filepath: str = r"C:\Users\juani\OneDrive\Escritorio\arxiv_data.csv", max_samples: int = 5000):
    print("Cargando el dataset de arXiv...")
    try:
        df = pd.read_csv(filepath)
    except Exception:
        df = pd.read_json(filepath, lines=True)

    print("Columnas encontradas:", df.columns.tolist())

    rename_map = {}
    if 'headline' in df.columns:
        rename_map['headline'] = 'title'
    if 'paperTitle' in df.columns:
        rename_map['paperTitle'] = 'title'
    if 'titles' in df.columns:
        rename_map['titles'] = 'title'
    if 'summary' in df.columns:
        rename_map['summary'] = 'abstract'
    if 'paperAbstract' in df.columns:
        rename_map['paperAbstract'] = 'abstract'
    if 'summaries' in df.columns:
        rename_map['summaries'] = 'abstract'

    df = df.rename(columns=rename_map)

    if not {'title', 'abstract'}.issubset(df.columns):
        raise ValueError(
            "El dataset no contiene las columnas 'title' y 'abstract'. "
            f"Columnas disponibles: {df.columns.tolist()}"
        )

    df = df[['title', 'abstract']].dropna().drop_duplicates()

    if len(df) > max_samples:
        df = df.sample(n=max_samples, random_state=42).reset_index(drop=True)

    df['title'] = df['title'].str.replace(r'\s+', ' ', regex=True).str.strip()
    df['abstract'] = df['abstract'].str.replace(r'\s+', ' ', regex=True).str.strip()
    df['text_to_embed'] = "Title: " + df['title'] + "\nAbstract: " + df['abstract']

    print(f"Corpus cargado con éxito. Total de documentos procesados: {len(df)}")
    return df

df_corpus = cargar_y_preprocesar_corpus()
df_corpus.head()

# Mostramos las primeras 5 filas para comprobar que todo se cargó bien
df_corpus.head()
df_corpus.head()

# Mostramos las primeras 5 filas para comprobar que todo se cargó bien
df_corpus.head()

Cargando el dataset de arXiv...
Columnas encontradas: ['titles', 'summaries', 'terms']
Corpus cargado con éxito. Total de documentos procesados: 5000


,title,abstract,text_to_embed
0,A Three-stage Approach for Segmenting Degraded...,"In this paper, we propose a SLaT (Smoothing, L...",Title: A Three-stage Approach for Segmenting D...
1,Human Recognition Using Face in Computed Tomog...,With the mushrooming use of computed tomograph...,Title: Human Recognition Using Face in Compute...
2,Sim-to-Real Transfer Learning using Robustifie...,Learning robot tasks or controllers using deep...,Title: Sim-to-Real Transfer Learning using Rob...
3,The Mapillary Traffic Sign Dataset for Detecti...,Traffic signs are essential map features globa...,Title: The Mapillary Traffic Sign Dataset for ...
4,Human Body Parts Tracking: Applications to Act...,"As cameras and computers became popular, the a...",Title: Human Body Parts Tracking: Applications...


In [34]:
# Instala las librerías necesarias para el procesamiento de embeddings y base vectorial
%pip install langchain langchain-community sentence-transformers faiss-cpu

Note: you may need to restart the kernel to use updated packages.


In [39]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# 1. Inicializar el modelo de embeddings de Hugging Face
print("Cargando modelo de embeddings (all-MiniLM-L6-v2)...")
embeddings_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'}, # Puedes usar 'cuda' si tienes GPU activa en tu entorno
    encode_kwargs={'normalize_embeddings': True}
)

# 2. Convertir las filas del dataframe en objetos Document de LangChain
print("Preparando documentos para la base vectorial...")
documentos = []
for idx, row in df_corpus.iterrows():
    doc = Document(
        page_content=row['text_to_embed'],
        metadata={
            "source_id": idx,
            "title": row['title'],
            "abstract": row['abstract']
        }
    )
    documentos.append(doc)

# 3. Construir e indexar la base de datos vectorial FAISS
print("Construyendo el índice vectorial FAISS en memoria... (Esto puede tomar un par de minutos)")
db_vectorial = FAISS.from_documents(documentos, embeddings_model)
print("¡Base de datos vectorial creada e indexada con éxito!")

Cargando modelo de embeddings (all-MiniLM-L6-v2)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5954.62it/s]


Preparando documentos para la base vectorial...
Construyendo el índice vectorial FAISS en memoria... (Esto puede tomar un par de minutos)
¡Base de datos vectorial creada e indexada con éxito!


In [40]:
# Guarda la base vectorial localmente para que Streamlit pueda usarla al instante
db_vectorial.save_local("faiss_index")
print("✅ Base vectorial guardada exitosamente en la carpeta 'faiss_index'.")

✅ Base vectorial guardada exitosamente en la carpeta 'faiss_index'.


In [ ]:
def recuperar_evidencias(query: str, vector_store, top_k: int = 3, similarity_threshold: float = 0.40):
    """
    Busca los documentos más relevantes. Si la similitud del mejor documento 
    no supera el umbral establecido, devuelve una lista vacía para alertar 
    de la falta de información relevante en el corpus.
    """
    # FAISS calcula la similitud de coseno y devuelve un score de relevancia de 0 a 1
    resultados = vector_store.similarity_search_with_relevance_scores(query, k=top_k)
    
    evidencias_filtradas = []
    for doc, score in resultados:
        if score >= similarity_threshold:
            evidencias_filtradas.append((doc, score))
            
    if not evidencias_filtradas:
        print(f"\n[Aviso]: La consulta '{query}' no tiene suficiente respaldo en el corpus (Similitud inferior a {similarity_threshold}).")
        return []
        
    return evidencias_filtradas

# --- HACER UNA PRUEBA REAL ---
consulta_prueba = "How is reinforcement learning used in robotics?"
print(f"Probando consulta: '{consulta_prueba}'")

evidencias = recuperar_evidencias(consulta_prueba, db_vectorial, top_k=3, similarity_threshold=0.40)

# Mostrar resultados
for i, (doc, score) in enumerate(evidencias):
    print(f"\n[Evidencia {i+1}] - Score de Similitud: {score:.4f}")
    print(f"Título: {doc.metadata['title']}")
    print(f"Resumen: {doc.metadata['abstract'][:250]}...\n")

Probando consulta: 'How is reinforcement learning used in robotics?'

[Evidencia 1] - Score de Similitud: 0.4745
Título: Reinforcement Learning in R
Resumen: Reinforcement learning refers to a group of methods from artificial intelligence where an agent performs learning through trial and error. It differs from supervised learning, since reinforcement learning requires no explicit labels; instead, the age...


[Evidencia 2] - Score de Similitud: 0.4743
Título: Importance of Environment Design in Reinforcement Learning: A Study of a Robotic Environment
Resumen: An in-depth understanding of the particular environment is crucial in reinforcement learning (RL). To address this challenge, the decision-making process of a mobile collaborative robotic assistant modeled by the Markov decision process (MDP) framewo...


[Evidencia 3] - Score de Similitud: 0.4284
Título: Reinforcement Learning for Robust Missile Autopilot Design
Resumen: Designing missiles' autopilot controllers has been a compl

In [ ]:
# Instalar el conector de Groq para LangChain
%pip install langchain-groq

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Instalar el conector oficial de Gemini para LangChain
%pip install -q langchain-google-genai

Note: you may need to restart the kernel to use updated packages.


In [ ]:
%pip install python-dotenv langchain-google-genai

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# ==========================================
# BLOQUE DE CÓDIGO: Generación RAG con Gemini (Local - VS Code)
# ==========================================

import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# --- PASO 1: Carga Segura de Variables de Entorno ---
# load_dotenv() busca el archivo .env en la carpeta actual y carga las variables
load_dotenv()

# Validamos que la API key se haya cargado correctamente en memoria
if "GOOGLE_API_KEY" in os.environ:
    print("✅ API Key de Gemini cargada exitosamente desde el archivo .env (Seguridad garantizada).")
else:
    print("❌ [Error]: No se encontró 'GOOGLE_API_KEY' en el archivo .env. Verifica la ruta y el nombre del archivo.")

# --- PASO 2: Inicialización de Gemini ---
# Usamos 'gemini-1.5-flash' por su excelente velocidad y rendimiento.
# Fijamos la temperatura en 0.0 para mitigar alucinaciones (comportamiento determinista).
try:
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",  # <--- CAMBIA ESTO AQUÍ (Antes era gemini-1.5-flash)
        temperature=0.0
    )
    print("✅ Modelo Gemini 2.5 Flash cargado exitosamente en memoria.")
except Exception as e:
    print(f"❌ Error al inicializar el modelo Gemini: {e}")
# --- PASO 3: Definición del Prompt de RAG con Instrucciones Estrictas ---
prompt_template = ChatPromptTemplate.from_messages([
    ("system", (
        "Eres un asistente de investigación científica altamente confiable especializado en artículos de arXiv.\n\n"
        "INSTRUCCIONES DE RESPUESTA:\n"
        "1. Responde a la pregunta del usuario utilizando ÚNICAMENTE el contexto provisto abajo.\n"
        "2. Si el contexto provisto está vacío, es irrelevante, o no contiene información suficiente para responder con certeza completa, "
        "debes responder EXACTAMENTE con la siguiente frase, sin añadir nada más:\n"
        "'Lo siento, pero el corpus de arXiv cargado no contiene suficiente información para responder a tu consulta.'\n"
        "3. No inventes datos, no alucines, ni uses conocimientos externos fuera de los documentos proporcionados.\n"
        "4. Redacta tu respuesta de manera profesional, estructurada y en español.\n\n"
        "CONTEXTO RECUPERADO DE ARXIV:\n"
        "{context}"
    )),
    ("human", "{question}")
])

# --- PASO 4: Función Principal del Sistema RAG ---
def ejecutar_rag(query: str, vector_store, llm_model, top_k: int = 3, similarity_threshold: float = 0.40):
    """
    Ejecuta el ciclo completo de RAG:
    1. Recupera evidencias usando el filtro de umbral (Requerimiento D).
    2. Si no hay evidencias, detiene el proceso y avisa al usuario.
    3. Si hay evidencias, formatea el contexto y genera la respuesta con el LLM.
    """
    # 1. Recuperación con filtro programático
    evidencias_filtradas = recuperar_evidencias(
        query=query, 
        vector_store=vector_store, 
        top_k=top_k, 
        similarity_threshold=similarity_threshold
    )
    
    # Manejo explícito si no hay evidencias suficientes (Filtro Programático)
    if not evidencias_filtradas:
        respuesta_vacia = "Lo siento, pero el corpus de arXiv cargado no contiene suficiente información para responder a tu consulta."
        return respuesta_vacia, []
        
    # 2. Preparación del contexto para el Prompt
    contexto_formateado = ""
    for i, (doc, score) in enumerate(evidencias_filtradas):
        contexto_formateado += f"\n--- Documento {i+1} (Similitud: {score:.4f}) ---\n"
        contexto_formateado += doc.page_content + "\n"
        
    # 3. Generación mediante la cadena de LangChain
    chain = prompt_template | llm_model | StrOutputParser()
    
    print("Generando respuesta con Gemini...")
    respuesta = chain.invoke({
        "context": contexto_formateado,
        "question": query
    })
    
    return respuesta, evidencias_filtradas

✅ API Key de Gemini cargada exitosamente desde el archivo .env (Seguridad garantizada).
✅ Modelo Gemini 2.5 Flash cargado exitosamente en memoria.


In [ ]:
#FASE PRUEBA CORRECTA
pregunta_existente = "How is reinforcement learning used in robotics?"
print(f"--- EJECUTANDO RAG CON GEMINI PARA: '{pregunta_existente}' ---")

respuesta, docs = ejecutar_rag(pregunta_existente, db_vectorial, llm)

print("\n=== RESPUESTA DEL LLM (GEMINI) ===")
print(respuesta)

--- EJECUTANDO RAG CON GEMINI PARA: 'How is reinforcement learning used in robotics?' ---
Generando respuesta con Gemini...

=== RESPUESTA DEL LLM (GEMINI) ===
El aprendizaje por refuerzo (RL) se utiliza en robótica de varias maneras, según el contexto proporcionado:

1.  **Asistencia Robótica Colaborativa:** Se emplea para estudiar el proceso de toma de decisiones de un asistente robótico móvil colaborativo. Este proceso se modela mediante el marco de procesos de decisión de Markov (MDP), y las combinaciones óptimas de estado-acción se calculan utilizando las ecuaciones de optimalidad de Bellman para determinar la política óptima.
2.  **Tareas Robóticas con Dominios de Acción Continuos:** El RL ha mostrado resultados interesantes en tareas robóticas que implican dominios de acción continuos.
3.  **Entrenamiento de Agentes para Control:** Se utiliza para entrenar un agente "model-free" que pueda controlar tareas robóticas, aunque aún se investiga cómo encontrar funciones de recompensa 

In [ ]:
# ==========================================
# BLOQUE DE CÓDIGO: Presentación de Evidencias
# ==========================================

def presentar_evidencias(evidencias_filtradas):
    """
    Formatea y presenta de manera visual y estructurada las evidencias (documentos)
    que sirvieron de contexto para alimentar al generador RAG.
    """
    if not evidencias_filtradas:
        print("\n" + "!" * 80)
        print("⚠️  AVISO: No se encontraron evidencias en el corpus que superen el umbral de similitud.")
        print("!" * 80)
        return

    print("\n" + "=" * 80)
    print("🔬  EVIDENCIAS CIENTÍFICAS RECUPERADAS (ARXIV)")
    print("=" * 80)
    
    for idx, (doc, score) in enumerate(evidencias_filtradas, 1):
        # Extraemos los metadatos que guardamos al indexar en el Requerimiento C
        titulo = doc.metadata.get("title", "Título no disponible").strip()
        abstract = doc.metadata.get("abstract", "Abstract no disponible").strip()
        id_original = doc.metadata.get("source_id", "N/A")
        
        print(f"\n[EVIDENCIA {idx}] — Score de Similitud Coseno: {score:.4f} (ID Interno: {id_original})")
        print(f"📌 TÍTULO: {titulo}")
        print(f"📖 ABSTRACT:\n{abstract}")
        print("-" * 80)

In [ ]:
# ==========================================
# BLOQUE DE CÓDIGO: Orquestador RAG + Evidencias
# ==========================================

def consultar_sistema_rag(query: str, vector_store, llm_model, top_k: int = 3, similarity_threshold: float = 0.40):
    """
    Función orquestadora que ejecuta la consulta, recupera datos,
    genera la respuesta del LLM y muestra las evidencias científicas de soporte.
    """
    print(f"\n🔍 Procesando consulta: '{query}'")
    
    # 1. Ejecutar el RAG (Recuperación + Generación)
    respuesta, evidencias = ejecutar_rag(
        query=query, 
        vector_store=vector_store, 
        llm_model=llm_model, 
        top_k=top_k, 
        similarity_threshold=similarity_threshold
    )
    
    # 2. Imprimir la respuesta generada por Gemini
    print(" RESPUESTA GENERADA POR GEMINI:")
    print(respuesta)
    
    # 3. Presentar las evidencias que sustentan la respuesta (Requerimiento F)
    presentar_evidencias(evidencias)

In [ ]:
# Consulta de prueba sobre un tema científico del dataset
consulta = "How is reinforcement learning used in robotics?"

# Ejecutamos todo el pipeline
consultar_sistema_rag(
    query=consulta, 
    vector_store=db_vectorial, 
    llm_model=llm, 
    top_k=3, 
    similarity_threshold=0.40
)


🔍 Procesando consulta: 'How is reinforcement learning used in robotics?'
Generando respuesta con Gemini...
 RESPUESTA GENERADA POR GEMINI:
El aprendizaje por refuerzo (RL) se utiliza en robótica de varias maneras, según el contexto proporcionado:

1.  **Asistencia Robótica Colaborativa:** Se emplea para estudiar el proceso de toma de decisiones de un asistente robótico móvil colaborativo. Este proceso se modela mediante el marco de procesos de decisión de Markov (MDP), y las combinaciones óptimas de estado-acción se calculan utilizando las ecuaciones de optimalidad de Bellman para determinar la política óptima.
2.  **Tareas Robóticas con Dominios de Acción Continuos:** El RL ha mostrado resultados interesantes en tareas robóticas que implican dominios de acción continuos.
3.  **Entrenamiento de Agentes para Control:** Se utiliza para entrenar un agente "model-free" que pueda controlar tareas robóticas, aunque aún se investiga cómo encontrar funciones de recompensa y estrategias de exp

In [42]:
%pip install gradio

Note: you may need to restart the kernel to use updated packages.
